In [1]:
import random
import pickle
import numpy as np
from collections import defaultdict
import scipy.sparse as sp

import os
import random
import pandas as pd
import json
import pickle
import gzip
from tqdm import tqdm

def load_pickle(filename):
    with open(filename, "rb") as f:
        return pickle.load(f)


def save_pickle(data, filename):
    with open(filename, "wb") as f:
        pickle.dump(data, f, protocol=pickle.HIGHEST_PROTOCOL)

def load_json(file_path):
    with open(file_path, "r") as f:
        return json.load(f)
    
def ReadLineFromFile(path):
    lines = []
    with open(path,'r') as fd:
        for line in fd:
            lines.append(line.rstrip('\n'))
    return lines

def save_pickle(data, filename):
    with open(filename, "wb") as f:
        pickle.dump(data, f, protocol=pickle.HIGHEST_PROTOCOL)

def parse(path):
    g = gzip.open(path, 'r')
    for l in g:
        # Convert bytes to string
        data_str = l.decode('utf-8')
        # Replace 'false' with 'False' and 'true' with 'True'
        data_str = data_str.replace('false', 'False').replace('true', 'True')

        # Parse the JSON string into a dictionary
        yield eval(data_str)
'''
Set seeds
'''
seed = 999
random.seed(seed)
np.random.seed(seed)

In [2]:
DATA_PATH = '../data/'

In [3]:
DATASET = 'yelp'

In [4]:
test_samples = ReadLineFromFile(os.path.join(DATA_PATH, DATASET, 'negative_samples.txt'))
len(test_samples)

30431

In [5]:
sequential_data = ReadLineFromFile(os.path.join(DATA_PATH, DATASET, 'sequential_data.txt'))
item_count = defaultdict(int)
user_items = defaultdict()

for line in sequential_data:
    user, items = line.strip().split(' ', 1)
    items = items.split(' ')
    items = [str(item) for item in items]
    user_items[user] = items
    for item in items:
        item_count[item] += 1

In [6]:
user_items['1']

['1',
 '2',
 '3',
 '4',
 '3',
 '5',
 '6',
 '3',
 '7',
 '8',
 '9',
 '10',
 '11',
 '2',
 '12',
 '12',
 '12',
 '12',
 '13',
 '14',
 '15',
 '16',
 '4',
 '2',
 '17',
 '18',
 '13',
 '19',
 '20',
 '21',
 '22',
 '23',
 '24',
 '25',
 '26',
 '27',
 '28',
 '29',
 '30',
 '31',
 '32',
 '33',
 '34',
 '35',
 '36',
 '37',
 '38',
 '39',
 '40',
 '41',
 '42',
 '43',
 '44',
 '45',
 '46',
 '47',
 '12',
 '48',
 '49',
 '50',
 '51',
 '52',
 '8',
 '14',
 '3',
 '53',
 '54',
 '55',
 '56']

In [7]:
all_item = list(item_count.keys())

In [8]:
all_item[:4]

['1', '2', '3', '4']

In [9]:
datamaps = load_json(os.path.join(DATA_PATH, DATASET, 'datamaps.json'))
user2id = datamaps['user2id']
item2id = datamaps['item2id']
user_list = list(datamaps['user2id'].keys())
item_list = list(datamaps['item2id'].keys())
id2item = datamaps['id2item']
id2user = datamaps['id2user']

In [11]:
list(id2user.keys())[:4]

['1', '2', '3', '4']

In [12]:
print("#samples:",len(test_samples[0].split(' ',1)[1].split(' ')))
test_samples[0].split(' ',1)[1].split(' ')[:10]

#samples: 99


['6482',
 '19925',
 '3415',
 '19024',
 '12325',
 '17809',
 '3897',
 '17873',
 '10579',
 '278']

In [13]:
train_negative = []
for user in tqdm(list(id2user.keys())):
    user_seq = user_items[user][:]
    user_seq = set([str(x) for x in user_seq])
    candidate_samples = []
    candidate_num = len(user_seq)
    already_samples = test_samples[int(user)-1].split(' ', 1)[1].split(' ')
    already_samples = set([str(x) for x in already_samples])
    while len(candidate_samples) < candidate_num:
        choices = [item for item in all_item if item not in user_seq]
        sample_ids = np.random.choice(choices, candidate_num, replace=False)
        sample_ids = [str(item) for item in sample_ids if (item not in user_seq)]
        sample_ids = [str(item) for item in sample_ids if (item not in already_samples)]
        sample_ids = [str(item) for item in sample_ids if (item not in candidate_samples)]
        candidate_samples.extend(sample_ids)
    candidate_samples = candidate_samples[:candidate_num]
    train_negative.append([user] + candidate_samples)
    

100%|██████████| 30431/30431 [01:01<00:00, 497.64it/s]


In [14]:
train_negative[0], user_items['1'], test_samples[0]

(['1',
  '12757',
  '2665',
  '19705',
  '6674',
  '1897',
  '18907',
  '10664',
  '12259',
  '4101',
  '7789',
  '19329',
  '18131',
  '328',
  '5024',
  '14766',
  '17826',
  '731',
  '16974',
  '10970',
  '19935',
  '10141',
  '15283',
  '19135',
  '13876',
  '6534',
  '10926',
  '9328',
  '16165',
  '10706',
  '8210',
  '7206',
  '216',
  '4708',
  '3915',
  '5937',
  '2973',
  '7570',
  '6283',
  '3125',
  '14623',
  '14859',
  '17473',
  '15907',
  '5581',
  '15244',
  '2407',
  '6603',
  '339',
  '18166',
  '9610',
  '19498',
  '12119',
  '2885',
  '9721',
  '9733',
  '18550'],
 ['1',
  '2',
  '3',
  '4',
  '3',
  '5',
  '6',
  '3',
  '7',
  '8',
  '9',
  '10',
  '11',
  '2',
  '12',
  '12',
  '12',
  '12',
  '13',
  '14',
  '15',
  '16',
  '4',
  '2',
  '17',
  '18',
  '13',
  '19',
  '20',
  '21',
  '22',
  '23',
  '24',
  '25',
  '26',
  '27',
  '28',
  '29',
  '30',
  '31',
  '32',
  '33',
  '34',
  '35',
  '36',
  '37',
  '38',
  '39',
  '40',
  '41',
  '42',
  '43',
  '44'

In [15]:
len(train_negative)

30431

# Check overlaps

In [16]:
test_negs = []
for idx in range(len(test_samples)):
    lst = test_samples[idx].split(' ')[1:]
    user = test_samples[idx].split(' ')[0]
    for val in lst:
        test_negs.append((int(user) - 1, int(val)))
len(test_negs)

3012669

In [17]:
test_negs[:10]

[(0, 6482),
 (0, 19925),
 (0, 3415),
 (0, 19024),
 (0, 12325),
 (0, 17809),
 (0, 3897),
 (0, 17873),
 (0, 10579),
 (0, 278)]

In [18]:
train_negs = []
for idx in range(len(train_negative)):
    lst = train_negative[idx]
    user = lst[0]
    items = lst[1:]
    for item in items:
        train_negs.append((int(user)-1,int(item)))

In [19]:
len(train_negs)

304524

In [20]:
train_negs[:10]

[(0, 12757),
 (0, 2665),
 (0, 19705),
 (0, 6674),
 (0, 1897),
 (0, 18907),
 (0, 10664),
 (0, 12259),
 (0, 4101),
 (0, 7789)]

In [21]:
inter = []
for user,items in user_items.items():
    new_user = int(user) - 1
    for item in items:
        item = int(item)
        inter.append((new_user,item))

In [22]:
len(inter)

316354

In [23]:
inter[:10]

[(0, 1),
 (0, 2),
 (0, 3),
 (0, 4),
 (0, 3),
 (0, 5),
 (0, 6),
 (0, 3),
 (0, 7),
 (0, 8)]

In [24]:
len(set(train_negs))

304524

In [25]:
len(set(test_negs))

3012669

In [26]:
len(set(inter))

304524

In [27]:
common1 = set(train_negs).intersection(set(test_negs))
len(common1)

0

In [28]:
common2 = set(train_negs).intersection(set(inter))
len(common2)

0

In [29]:
common2

set()

In [31]:
save_pickle(train_negative,os.path.join(DATA_PATH,DATASET,'train-negatives.pkl'))